# Week 4 — Supply Chain Monitoring



In [2]:
!pip -q install pyspark==3.5.1 delta-spark==3.1.0

In [3]:
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, lower, to_date, when, count, sum as spark_sum, avg, datediff, current_date
from delta import configure_spark_with_delta_pip
import os
import shutil

In [4]:
uploaded = files.upload()
required_files = ('suppliers.csv', 'orders.csv')
missing_files = set(required_files) - set(uploaded.keys())
if missing_files:
    raise ValueError(f"Upload these files: {sorted(missing_files)}")

Saving orders.csv to orders.csv
Saving suppliers.csv to suppliers.csv


In [6]:
builder = SparkSession.builder.appName("Week4SupplyChainETL").config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
spark = configure_spark_with_delta_pip(builder).getOrCreate()

## 1. Load source data

In [7]:
reference_raw = spark.read.option("header", True).csv("/content/suppliers.csv")
raw = spark.read.option("header", True).csv("/content/orders.csv")
reference = reference_raw
suppliers_raw = reference_raw
raw.show(5, truncate=False)
reference_raw.show(5, truncate=False)

+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|shipment_id|supplier_id|order_date|expected_delivery_date|actual_delivery_date|quantity|shipment_cost|status    |
+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|SH001      |S001       |2026-01-01|2026-01-08            |2026-01-08          |37      |1751         |IN_TRANSIT|
|SH002      |S002       |2026-01-02|2026-01-09            |2026-01-09          |54      |2002         |PENDING   |
|SH003      |S003       |2026-01-03|2026-01-10            |2026-01-13          |71      |2253         |RECEIVED  |
|SH004      |S004       |2026-01-04|2026-01-11            |2026-01-11          |88      |2504         |IN_TRANSIT|
|SH005      |S005       |2026-01-05|2026-01-12            |2026-01-12          |105     |2755         |PENDING   |
+-----------+-----------+----------+----------------------+--------------------+

## 2. Clean the reference data

In [8]:
reference = reference.select(trim("supplier_id").alias("supplier_id"), trim("supplier_name").alias("supplier_name"), trim("city").alias("city"), upper(trim("region")).alias("region")).filter((col("supplier_id") != "") & col("supplier_id").isNotNull() & (col("supplier_name") != "")).dropDuplicates(["supplier_id"])
reference.show(truncate=False)
print('Reference rows after cleaning:', reference.count())

+-----------+------------------+---------+------+
|supplier_id|supplier_name     |city     |region|
+-----------+------------------+---------+------+
|S001       |Apex Components   |Chennai  |SOUTH |
|S002       |Global Supplies   |Mumbai   |WEST  |
|S003       |Rapid Logistics   |Hyderabad|SOUTH |
|S004       |Prime Materials   |Delhi    |NORTH |
|S005       |Eastern Traders   |Kolkata  |EAST  |
|S006       |Reliable Parts    |Bengaluru|SOUTH |
|S007       |Coastal Wholesale |Kochi    |SOUTH |
|S008       |Metro Distributors|Pune     |WEST  |
+-----------+------------------+---------+------+

Reference rows after cleaning: 8


## 3. Clean and transform transactional data

In [9]:
data = raw.select(trim("shipment_id").alias("shipment_id"), trim("supplier_id").alias("supplier_id"), to_date(trim("order_date")).alias("order_date"), to_date(trim("expected_delivery_date")).alias("expected_delivery_date"), to_date(trim("actual_delivery_date")).alias("actual_delivery_date"), col("quantity").cast("integer").alias("quantity"), col("shipment_cost").cast("double").alias("shipment_cost"), upper(trim("status")).alias("status"))
data = data.filter((col("shipment_id") != "") & col("shipment_id").isNotNull() & (col("supplier_id") != "") & col("order_date").isNotNull() & col("expected_delivery_date").isNotNull() & (col("quantity") > 0) & (col("shipment_cost") > 0)).dropDuplicates(["shipment_id"])
data = data.withColumn("status", when(col("status").isNull() | (col("status") == ""), "PENDING").otherwise(col("status")))
data.show(truncate=False)
print('Transaction rows after cleaning:', data.count())

+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|shipment_id|supplier_id|order_date|expected_delivery_date|actual_delivery_date|quantity|shipment_cost|status    |
+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|SH001      |S001       |2026-01-01|2026-01-08            |2026-01-08          |37      |1751.0       |IN_TRANSIT|
|SH002      |S002       |2026-01-02|2026-01-09            |2026-01-09          |54      |2002.0       |PENDING   |
|SH003      |S003       |2026-01-03|2026-01-10            |2026-01-13          |71      |2253.0       |RECEIVED  |
|SH004      |S004       |2026-01-04|2026-01-11            |2026-01-11          |88      |2504.0       |IN_TRANSIT|
|SH005      |S005       |2026-01-05|2026-01-12            |2026-01-12          |105     |2755.0       |PENDING   |
|SH006      |S006       |2026-01-06|2026-01-13            |NULL                |

## 4. Join and derive business metrics

In [10]:
details = data.join(reference, "supplier_id", "inner").withColumn("delay_days", when(col("actual_delivery_date").isNotNull(), datediff(col("actual_delivery_date"), col("expected_delivery_date"))).otherwise(datediff(current_date(), col("expected_delivery_date")))).withColumn("is_delayed", col("delay_days") > 0)
summary = details.groupBy("supplier_id", "supplier_name", "region").agg(count("shipment_id").alias("total_shipments"), spark_sum(when(col("is_delayed"),1).otherwise(0)).alias("delayed_shipments"), spark_sum("shipment_cost").alias("total_shipment_cost"), avg("delay_days").alias("average_delay_days"))
details.show(10, truncate=False)
summary.show(truncate=False)

+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+------------------+---------+------+----------+----------+
|supplier_id|shipment_id|order_date|expected_delivery_date|actual_delivery_date|quantity|shipment_cost|status    |supplier_name     |city     |region|delay_days|is_delayed|
+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+------------------+---------+------+----------+----------+
|S001       |SH001      |2026-01-01|2026-01-08            |2026-01-08          |37      |1751.0       |IN_TRANSIT|Apex Components   |Chennai  |SOUTH |0         |false     |
|S002       |SH002      |2026-01-02|2026-01-09            |2026-01-09          |54      |2002.0       |PENDING   |Global Supplies   |Mumbai   |WEST  |0         |false     |
|S003       |SH003      |2026-01-03|2026-01-10            |2026-01-13          |71      |2253.0       |RECEIVED  |Rapid Logistics   |Hy

## 5. Write cleaned data to CSV and Delta Lake

In [11]:
output_root = "/content/week4_supply_chain_monitoring_outputs"
os.makedirs(output_root, exist_ok=True)
data.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/cleaned_data_csv")
details.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/joined_details_csv")
summary.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/business_summary_csv")
data.write.format("delta").mode("overwrite").save(f"{output_root}/cleaned_data_delta")
details.write.format("delta").mode("overwrite").save(f"{output_root}/joined_details_delta")
summary.write.format("delta").mode("overwrite").save(f"{output_root}/business_summary_delta")

## 6. Read back and validate Delta data

In [12]:
clean_delta = spark.read.format("delta").load(f"{output_root}/cleaned_data_delta")
detail_delta = spark.read.format("delta").load(f"{output_root}/joined_details_delta")
summary_delta = spark.read.format("delta").load(f"{output_root}/business_summary_delta")
print("Clean rows:", clean_delta.count())
print("Joined rows:", detail_delta.count())
print("Summary rows:", summary_delta.count())
clean_delta.show(5, truncate=False)

Clean rows: 32
Joined rows: 31
Summary rows: 8
+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|shipment_id|supplier_id|order_date|expected_delivery_date|actual_delivery_date|quantity|shipment_cost|status    |
+-----------+-----------+----------+----------------------+--------------------+--------+-------------+----------+
|SH001      |S001       |2026-01-01|2026-01-08            |2026-01-08          |37      |1751.0       |IN_TRANSIT|
|SH002      |S002       |2026-01-02|2026-01-09            |2026-01-09          |54      |2002.0       |PENDING   |
|SH003      |S003       |2026-01-03|2026-01-10            |2026-01-13          |71      |2253.0       |RECEIVED  |
|SH004      |S004       |2026-01-04|2026-01-11            |2026-01-11          |88      |2504.0       |IN_TRANSIT|
|SH005      |S005       |2026-01-05|2026-01-12            |2026-01-12          |105     |2755.0       |PENDING   |
+-----------+-----------+--------

## 7. Download generated deliverables

In [13]:
shutil.make_archive("/content/week4_supply_chain_monitoring_outputs", "zip", output_root)
files.download("/content/week4_supply_chain_monitoring_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
spark.stop()